# 01 · Análisis exploratorio (EDA)

## ¿Qué hace el EDA y por qué se hace ANTES de entrenar?
El EDA responde una sola pregunta: **¿estos datos me dejan aprender lo que quiero aprender?**
No es "mirar gráficos": cada paso busca una trampa concreta que, si no se detecta ahora, infla los resultados o
produce un modelo que falla en la calle.

| Paso | Pregunta | Trampa que busca |
|---|---|---|
| 1. Inventario | ¿qué tengo y de dónde viene? | fuentes con una sola clase (atajo de la fuente) |
| 2. Tamaño y color | ¿las imágenes son comparables? | tamaños mezclados, tintes distintos por fuente |
| 3. Brillo | ¿las clases se distinguen por la luz? | **atajo del brillo** |
| 4. Duplicados | ¿hay imágenes repetidas? | fuga por copias |
| 5. Adyacencia | ¿archivos consecutivos son de la misma foto? | decide cómo agrupar |
| 6. División | ¿cómo separo entrenamiento/validación/prueba? | **fuga por hermanos** (el 99,93 % inflado) |
| 7. Línea base | ¿cuánto logra un método simple? | un 99 % sin referencia no significa nada |

In [ ]:
# ===================== CELDA DE ARRANQUE: no hay que editar nada =====================
# Monta tu Drive, encuentra sola la carpeta del proyecto y Surface Crack, y deja todo listo.
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys
_raiz = "/content/drive/MyDrive"
_cands = sorted(glob.glob(f"{_raiz}/*/src/datos.py") + glob.glob(f"{_raiz}/*/*/src/datos.py"),
                key=os.path.getmtime, reverse=True)
if not _cands:
    raise FileNotFoundError("No encontré la carpeta del proyecto NUEVO en tu Drive (busco una carpeta que tenga src/datos.py). "
                            "Subí la carpeta del zip nuevo a tu Drive y volvé a ejecutar esta celda.")
PROYECTO = os.path.dirname(os.path.dirname(_cands[0]))
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)
print("Proyecto:", PROYECTO)
!pip -q install scikit-image scikit-learn pandas matplotlib

from src import colab
RUTA_DATOS = colab.preparar(PROYECTO, _raiz)       # Surface Crack (zip -> disco local, o carpeta en Drive)

import json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from src import config, datos, eda, evaluar

## Paso 1 · Inventario
**Qué hace:** recorre las carpetas y arma una tabla con una fila por imagen: ruta, etiqueta (1 = con grieta),
fuente, **grupo** y orden. El *grupo* es la idea clave: las imágenes "hermanas" (recortes de la misma foto, fotos
de la misma pared) comparten grupo y **nunca podrán separarse** entre entrenamiento y prueba.

**Qué mirar:** el balance entre clases y la **auditoría de fuentes**. Si una fuente tiene solo una clase, el modelo
podría aprender "foto de celular = sin grieta" en vez de aprender qué es una grieta.

In [ ]:
fuentes = {
    "surface_crack": {"con_grieta": f"{RUTA_DATOS}/Positive", "sin_grieta": f"{RUTA_DATOS}/Negative",
                      "grupos": "bloque", "tam_bloque": config.TAM_BLOQUE},
    # --- Descomentá cada una cuando tengas las imágenes (ver docs/CORRECCION_FALSO_POSITIVO.md) ---
    # "negativos_dificiles": {"sin_grieta": f"{PROYECTO}/data/dificiles", "grupos": "subcarpeta"},
    # "campo":               {"con_grieta": f"{PROYECTO}/data/campo/con", "sin_grieta": f"{PROYECTO}/data/campo/sin", "grupos": "subcarpeta"},
    # SDNET2018: REPARTILO, no lo uses entero para entrenar. Verificá los nombres de carpeta al descargarlo.
    #   para ENTRENAR (paredes, lo más parecido a tu caso):
    # "sdnet_paredes":       {"con_grieta": f"{PROYECTO}/data/sdnet/W/CW", "sin_grieta": f"{PROYECTO}/data/sdnet/W/UW", "grupos": "archivo"},
    #   para PROBAR fuera de dominio (otra categoría entera; se aparta con reservar_fuente):
    # "sdnet_fuera":         {"con_grieta": f"{PROYECTO}/data/sdnet/P/CP", "sin_grieta": f"{PROYECTO}/data/sdnet/P/UP", "grupos": "archivo"},
}

df = datos.construir_inventario(fuentes)
print(f"{len(df):,} imágenes · {df['grupo'].nunique():,} grupos")
tabla, avisos = datos.auditoria_fuentes(df)
display(tabla)
for a in avisos:
    print("⚠", a)

## Paso 1b · Apartar lo que NO se usa para entrenar
**Qué hace:** saca del inventario las fuentes reservadas para evaluar (SDNET fuera de dominio y tus fotos de campo) y
las guarda aparte. **Se hace antes de dividir**, para que ninguna imagen de evaluación llegue al entrenamiento ni a la
validación. Si entrenaras con ellas, la prueba fuera de dominio dejaría de ser honesta.

In [ ]:
EVALUACION = ["sdnet_fuera", "campo"]          # fuentes que el modelo NUNCA ve al entrenar
fuera = df[df["fuente"].isin(EVALUACION)]
df = df[~df["fuente"].isin(EVALUACION)].reset_index(drop=True)
Path("results").mkdir(exist_ok=True)
fuera.to_csv("results/fuera_de_dominio.csv", index=False)
print(f"Para entrenar/validar/probar: {len(df):,} imágenes · Apartadas para evaluar fuera de dominio: {len(fuera):,}")

## Paso 2 · Tamaño y color
**Qué hace:** mide el tamaño (ancho × alto × modo) y la media de cada canal RGB en una muestra, **por fuente**.

**Qué mirar:** todas las imágenes de una fuente deberían tener el mismo tamaño. Concreto gris → R ≈ G ≈ B; una
fuente con un tinte distinto (pared pintada, filtro cálido) se nota aquí.

In [ ]:
for fuente, sub in df.groupby("fuente"):
    rutas = sub["ruta"].tolist()
    print(f"\n== {fuente} ==")
    print("tamaños:", dict(eda.resumen_tamanos(rutas, n=300)))
    print("color medio:", {k: round(v, 1) for k, v in eda.resumen_color(rutas, n=300).items()})

## Paso 3 · Brillo por imagen (el atajo del brillo)
**Qué hace:** calcula el brillo promedio (0–255) de **cada** imagen y compara las dos clases. Es tu función
`brillo_promedio`, pero guardando **una lista** (un valor por imagen) en vez de un acumulador (un solo número).

**Cómo leerlo**
* **Cohen d**: separación entre clases en desvíos estándar (0,2 chico · 0,5 mediano · 0,8 grande).
* **Clasificador tonto**: la exactitud MÁXIMA que lograría alguien que *solo* mirara el brillo. Si da ~65 %, el
  atajo es débil. Si diera 95 %, el modelo seguramente lo estaría usando.

**Qué decidir:** si el atajo es débil, la aumentación de brillo basta; si es fuerte, hay que corregir los datos.

In [ ]:
N = 4000   # muestra por clase para ir rápido; poné None para usar todas
pos = df[df.etiqueta == 1]["ruta"].tolist()[:N]
neg = df[df.etiqueta == 0]["ruta"].tolist()[:N]
b_pos, b_neg = eda.brillo_por_imagen(pos), eda.brillo_por_imagen(neg)
print(f"brillo medio  con grieta: {b_pos.mean():.1f} | sin grieta: {b_neg.mean():.1f}")
print(f"Cohen d = {eda.cohen_d(b_neg, b_pos):.2f}")
tonto = eda.clasificador_tonto_brillo(b_pos, b_neg)
print(f"Un clasificador que SOLO mira el brillo acierta como máximo {100*tonto['exactitud']:.1f} % (azar = {100*tonto['azar']:.0f} %)")
Path("results").mkdir(exist_ok=True)
eda.figura_brillo(b_pos, b_neg, "results/histograma_brillo.png");

## Paso 4 · Duplicados (huella perceptual)
**Qué hace:** reduce cada imagen a una huella de 64 bits (gris → 8×8 → cada píxel contra el promedio). Dos
imágenes con la misma huella son casi-copias. La huella **no cambia** si se aclara la imagen uniformemente.

**Límite:** es una cota inferior. No detecta *recortes distintos de la misma foto*: eso lo resuelve el paso 5.

In [ ]:
muestra = df.sample(min(4000, len(df)), random_state=config.SEMILLA)
dup = datos.encontrar_duplicados(muestra["ruta"].tolist())
redundantes = sum(len(v) - 1 for v in dup.values())
print(f"{len(dup)} grupos de duplicados · {redundantes} imágenes redundantes ({100*redundantes/len(muestra):.2f} % de la muestra)")

## Paso 5 · ¿Los archivos consecutivos son de la misma foto?
**Por qué importa:** Surface Crack son ~40.000 recortes salidos de 458 fotos. Si los archivos consecutivos salen de
la misma foto, partir por **bloques de ~44** mantiene juntos a los hermanos. Pero eso es una **hipótesis**.

**Qué hace:** compara la distancia (en color medio) entre archivos consecutivos con la de pares al azar.
* razón **< 0,7** → los consecutivos se parecen → los bloques sirven.
* razón **≈ 1** → no hay relación → usar agrupación por similitud (`asignar_grupos_cluster`).

In [ ]:
# se calculan sobre el inventario ordenado (la prueba necesita el orden de los archivos)
sub = df.groupby(["fuente", "etiqueta"], group_keys=False).apply(lambda g: g.head(3000)).reset_index(drop=True)
rasgos = datos.extraer_rasgos(sub)
adyacencia = datos.prueba_adyacencia(sub, rasgos)
print(adyacencia)

**Decisión.** Mirá `razon` y `veredicto`. Si dice que los bloques NO sirven, ejecutá la celda siguiente (agrupar
por similitud). Si sirven, saltala.

In [ ]:
USAR_CLUSTER = adyacencia["razon"] >= 0.9      # ← cambialo a mano si querés forzarlo
if USAR_CLUSTER:
    rasgos_todos = datos.extraer_rasgos(df)
    df = datos.asignar_grupos_cluster(df, rasgos_todos, imagenes_por_grupo=config.TAM_BLOQUE)
    print("Grupos reasignados por similitud:", df["grupo"].nunique())
else:
    print("Se mantienen los bloques:", df["grupo"].nunique(), "grupos")

## Paso 6 · División sin fuga
**Qué hace:** separa en entrenamiento / validación / prueba **por grupos enteros**. Verifica con un `assert` que
ningún grupo ni imagen aparece en dos particiones.

**Por qué:** dividir imagen por imagen mezclaría recortes del mismo muro en entrenamiento y prueba. El modelo
"se examinaría con hermanos de lo que ya vio" y el resultado saldría inflado. La celda siguiente **lo demuestra**
con tus datos.

In [ ]:
from sklearn.model_selection import train_test_split
tr0, te0 = train_test_split(df, test_size=0.15, random_state=config.SEMILLA)
mezclados = len(set(tr0["grupo"]) & set(te0["grupo"]))
print(f"División imagen por imagen (la anterior): {mezclados} grupos aparecen en entrenamiento Y en prueba  ← FUGA")

tr, va, te = datos.dividir(df)
print("División por grupos: sin fuga (verificada)")
display(datos.resumen_particiones(entrenamiento=tr, validacion=va, prueba=te))
print("huellas compartidas train↔test:", datos.huellas_compartidas(tr.sample(min(2000, len(tr)), random_state=1), te.sample(min(2000, len(te)), random_state=1)))

## Paso 7 · Línea base (¿cuánto logra un método simple?)
**Qué hace:** ajusta tres métodos **sin red neuronal** con el entrenamiento y los mide en prueba: umbral de brillo,
umbral de bordes y regresión logística sobre 5 rasgos. Esta es la tabla que el proyecto tenía marcada
como *[pendiente]*.

**Para qué:** sin una línea base, "99 %" no significa nada. La red debe superar claramente a estos métodos
y, sobre todo, **medirse con la misma división sin fuga**.

In [ ]:
from src import linea_base
sub_tr = tr.sample(min(6000, len(tr)), random_state=config.SEMILLA).reset_index(drop=True)
sub_te = te.sample(min(3000, len(te)), random_state=config.SEMILLA).reset_index(drop=True)
tabla_lb = linea_base.tabla_linea_base(sub_tr, sub_te)
display(tabla_lb)
tabla_lb.to_csv("results/linea_base.csv", index=False)

## Guardar las particiones
Se guardan para que **entrenamiento y evaluación usen exactamente las mismas** (reproducibilidad).

In [ ]:
tr.to_csv("results/particion_entrenamiento.csv", index=False)
va.to_csv("results/particion_validacion.csv", index=False)
te.to_csv("results/particion_prueba.csv", index=False)
df.to_csv("results/inventario.csv", index=False)
print("Guardado en results/. Siguiente: 02_Entrenamiento.ipynb")